# NLGCL Training on Kaggle

This notebook sets up the environment and runs the NLGCL model training.

In [ ]:
# 1. Install Dependencies
!pip install -q recbole colorlog tensorboard colorama pyyaml pandas hyperopt==0.2.5

# 2. Install PyTorth Geometric and related libraries
# Note: Kaggle usually runs on Linux with CUDA. We adapt the installation for that environment.
import torch

def format_pytorch_version(version):
    return version.split('+')[0]

TORCH_version = torch.__version__
CUDA_version = torch.version.cuda

print(f"Torch version: {TORCH_version}")
print(f"CUDA version: {CUDA_version}")

TORCH = format_pytorch_version(TORCH_version)
CUDA = "cu" + CUDA_version.replace(".", "")

print(f"Installing for Torch {TORCH} and CUDA {CUDA}")

!pip install -q torch-geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

# 3. Clone Repository
!git clone https://github.com/yangzeha/NLGCL.git
%cd NLGCL

# 4. Patch Missing Files (Important step to fix the ImportError)
import os

# List of files that were missing in the local environment and caused errors
# We create dummy classes for them so the __init__.py imports don't fail.
missing_modules = [
    ("lightgcn", "LightGCN"),
    ("hmlet", "HMLET"),
    ("ncl", "NCL"),
    ("ngcf", "NGCF"),
    ("sgl", "SGL"),
    ("lightgcl", "LightGCL"),
    ("simgcl", "SimGCL"),
    ("xsimgcl", "XSimGCL"),
    ("directau", "DirectAU"),
    ("ssl4rec", "SSL4REC"),
    ("dccf", "DCCF"),
    ("l2cl", "L2CL")
]

base_path = "recbole_gnn/model/general_recommender"

for module_name, class_name in missing_modules:
    file_path = os.path.join(base_path, f"{module_name}.py")
    # Only create if it doesn't exist (it likely won't)
    if not os.path.exists(file_path):
        print(f"Creating dummy file for: {file_path}")
        with open(file_path, 'w') as f:
            f.write(f"class {class_name}:\n    pass\n")
    else:
        print(f"File exists: {file_path}")

# 5. Run Training
# QB-Video settings based on author's recommendation:
# n_layers = 4, reg_weight = 0.0001, cl_temp = 0.2, alpha = 0.6, cl_reg = 5e-5
!python main.py --dataset QB-video --model NLGCL --n_layers=4 --reg_weight=1e-4 --cl_temp=0.2 --alpha=0.6 --cl_reg=5e-5

In [ ]:
import os
import shutil

# Dataset Handling
dataset_name = "QB-video"
kaggle_input_path = f"/kaggle/input/{dataset_name.lower()}"
target_path = f"dataset/{dataset_name}"

if not os.path.exists('dataset'):
    os.makedirs('dataset')

if os.path.exists(kaggle_input_path):
    print(f"Found dataset at {kaggle_input_path}")
    if os.path.exists(target_path):
        shutil.rmtree(target_path)
    # create symlink or copy
    # Symlink might not work for some file systems in Kaggle wrappers, copy is safer but slower.
    # We'll try symlink first, if fails, copy.
    try:
        os.symlink(kaggle_input_path, target_path)
        print(f"Created symlink to {target_path}")
    except OSError:
        shutil.copytree(kaggle_input_path, target_path)
        print(f"Copied dataset to {target_path}")
else:
    print(f"Dataset not found at {kaggle_input_path}. Please follow the instructions to upload the dataset.")
    # Attempt to create an empty folder so execution doesn't fail strictly on path check (it will fail later if files missing)
    if not os.path.exists(target_path):
        os.makedirs(target_path)


## 3. Dataset Preparation
**Important:** The QB-Video dataset is not included in the repository and must be downloaded manually or added as a Kaggle Dataset.

### Option A: Use Kaggle Dataset (Recommended)
1. Download the QB-Video dataset from [RecBole Google Drive](https://drive.google.com/drive/folders/1so0lckI6N6_niVEYaBu-LIcpOdZf99kj?usp=sharing).
2. Look for the `QB-video` folder.
3. In Kaggle, click "Add Data" -> "Upload" -> Upload the files/folder.
4. Rename the dataset to `qb-video` for easier access.

### Option B: Download via Code (If a direct link is available)
The code below attempts to link the dataset if it exists in Kaggle input.
